In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
table = dbutils.widgets.get("table") 

experiment_path = "/Workspace/Shared/MLflow_Experiments/proyecto_1_demo"
registered_model_rf = f"{catalog}.{schema}.modelo_rf_classifier"
registered_model_xgb = f"{catalog}.{schema}.modelo_xgb_classifier"

dataset_path = f"{catalog}.{schema}.{table}"
print(dataset_path)

In [0]:
import mlflow
import pandas as pd
import numpy as np

In [0]:
class FeatureEngineering:
    """Feature Engineer.

        Esta clase se encarga de la preparación de características específicas para el análisis de datos.

    Attributes:
        df (pd.DataFrame): El DataFrame que contiene los datos.
    """
    def __init__(self, df: pd.DataFrame):
        self.df = df
    
    def _list_target_numeric_and_categorical_features (self, df: pd.DataFrame):
        """ Obtiene la lista de variables categoricas y numericas de un dataframe

        Args:
             df (pd.DataFrame): El DataFrame que contiene los datos.

        Returns:
            target_variable: variable a predecir
            numeric_features: Lista con las variables numericas
            categorical_features: Lista con las variables categoricas
        """
        target_variable = 'respuesta'
        numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
        numeric_features = [col for col in numeric_cols if col != target_variable]
        categorical_cols = df.select_dtypes(include=['object', 'category']).columns
        categorical_features = [col for col in categorical_cols]
        
        return target_variable, numeric_features, categorical_features


    def prepare_features_only_energy(self) -> pd.DataFrame:
        """ Prepara las características específicas para el análisis de datos de energía.

        Returns:
            pd.DataFrame: El DataFrame con las características preparadas de datos de energía.
        """
        df_energy = self.df[self.df['tipo_servicio'] == 'Energia']
        target_variable, numeric_features, categorical_features = self._list_target_numeric_and_categorical_features(df_energy)


        return df_energy,target_variable, numeric_features, categorical_features

In [0]:
df_spark = spark.table(dataset_path)
df_pandas = df_spark.toPandas()

feature_engineering = FeatureEngineering(df_pandas)
df_fe, target_variable, numeric_features, categorical_features = feature_engineering.prepare_features_only_energy()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Target variable:", target_variable)

In [0]:
mlflow.set_experiment(experiment_path)

In [0]:
import logging

import pandas as pd
import optuna
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score, roc_auc_score)
from optuna.integration import MLflowCallback

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class TrainModel:
    def __init__(self, df: pd.DataFrame, 
                 numeric_features: list, 
                 categorical_features: list, 
                 target_column: str,
                 model_class,
                 test_size: float = 0.2,
                 model_params: dict = None,
                 param_distributions: dict = None,
                 n_trials: int = 10,
                 optimization_metric: str = "accuracy",
                 mlflow_setup=None,
                 mlflow_registered_model_name: str = None,
                 mlflow_experiment_path: str = None):
        
        self.df = df
        self.numeric_features = numeric_features
        self.categorical_features = categorical_features
        self.target_column = target_column
        self.model_class = model_class
        self.test_size = test_size
        self.model_params = model_params
        self.param_distributions = param_distributions or {}
        self.n_trials = n_trials
        self.optimization_metric = optimization_metric
        self.mlflow_setup = mlflow_setup
        self.mlflow_registered_model_name = mlflow_registered_model_name
        self.mlflow_experiment_path = mlflow_experiment_path

    def train_test_split(self):
        X = self.df.drop(columns=[self.target_column])
        y = self.df[self.target_column]
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=self.test_size, random_state=42
        )
        return X_train, X_test, y_train, y_test
    
    def create_pipeline_numeric(self):
        numeric_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')), 
            ('scaler', StandardScaler())
        ])
        return numeric_transformer
    

    def create_pipeline_categorical(self):
        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(
                strategy='constant', fill_value='Unknown'
            )),
            ('onehot', OneHotEncoder(
                drop='first', 
                sparse_output=False, 
                handle_unknown='ignore'
            ))
        ])
        return categorical_transformer

    def create_preprocessor(self):
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', self.create_pipeline_numeric(), 
                 self.numeric_features),
                ('cat', self.create_pipeline_categorical(), 
                 self.categorical_features)
            ]
        )
        return preprocessor
    
    def create_pipeline_train(self, model):
        """Create pipeline with a specific model instance."""
        pipeline = Pipeline(steps=[
            ('preprocessor', self.create_preprocessor()),
            ('classifier', model)
        ])
        return pipeline
    
    def create_objective(self, X_train, X_test, y_train, y_test):
        """Create an Optuna objective function for hyperparameter optimization."""
        
        def objective(trial):
            # Suggest hyperparameters from the provided distributions
            params = {}
            for param_name, param_config in self.param_distributions.items():
                param_type = param_config[0]
                
                if param_type == 'float':
                    min_val, max_val = param_config[1], param_config[2]
                    log = param_config[3] if len(param_config) > 3 else False
                    params[param_name] = trial.suggest_float(
                        param_name, min_val, max_val, log=log
                    )
                elif param_type == 'int':
                    min_val, max_val = param_config[1], param_config[2]
                    params[param_name] = trial.suggest_int(
                        param_name, min_val, max_val
                    )
                elif param_type == 'categorical':
                    choices = param_config[1]
                    params[param_name] = trial.suggest_categorical(
                        param_name, choices
                    )
            
            # Merge with fixed params if provided
            all_params = {**self.model_params, **params}
            
            # Create model instance with suggested parameters
            model = self.model_class(**all_params)
            
            # Create and train pipeline
            pipeline = self.create_pipeline_train(model)
            
            # Fit the pipeline
            pipeline.fit(X_train, y_train)
            
            # Calculate the optimization metric
            if self.optimization_metric == 'accuracy':
                y_pred = pipeline.predict(X_test)
                score = accuracy_score(y_test, y_pred)
            elif self.optimization_metric == 'f1':
                y_pred = pipeline.predict(X_test)
                score = f1_score(y_test, y_pred, average='weighted')
            elif self.optimization_metric == 'precision':
                y_pred = pipeline.predict(X_test)
                score = precision_score(y_test, y_pred, average='weighted', zero_division=0)
            elif self.optimization_metric == 'recall':
                y_pred = pipeline.predict(X_test)
                score = recall_score(y_test, y_pred, average='weighted', zero_division=0)
            elif self.optimization_metric == 'roc_auc':
                # For ROC AUC, we need probability predictions
                if hasattr(pipeline, 'predict_proba'):
                    y_pred_proba = pipeline.predict_proba(X_test)
                    # For binary classification, use the positive class probability
                    if y_pred_proba.shape[1] == 2:
                        y_pred_proba = y_pred_proba[:, 1]
                    score = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
                else:
                    # Fallback to accuracy if model doesn't support predict_proba
                    y_pred = pipeline.predict(X_test)
                    score = accuracy_score(y_test, y_pred)
            else:
                raise ValueError(f"Unknown optimization metric: {self.optimization_metric}")
            
            return score
        
        return objective
    
    def train_with_optuna(self):
        """
        Train the model with Optuna hyperparameter optimization and MLflow tracking.

        Returns:
            best_pipeline: Best trained sklearn pipeline
            best_run_id: MLflow run ID for the best model
            study: Optuna study object with optimization results
        """
        # Split data once for all trials
        X_train, X_test, y_train, y_test = self.train_test_split()
        
        self.mlflow_setup.set_experiment(self.mlflow_experiment_path)

        # Create Optuna study
        study = optuna.create_study(
            direction='maximize',
            study_name=self.mlflow_experiment_path
        )
        
        # Create objective function
        objective = self.create_objective(X_train, X_test, y_train, y_test)
        
        # Set up MLflow callback for Optuna
        mlflow_callback = MLflowCallback(
            tracking_uri=self.mlflow_setup.get_tracking_uri(),
            metric_name=self.optimization_metric
        )
        
         # Run optimization
        logger.info(f"Starting Optuna optimization with {self.n_trials} trials...")
        logger.info(f"Optimizing for: {self.optimization_metric}")
        logger.info(f"Model type: {self.model_class.__name__}")
        
        study.optimize(
            objective, 
            n_trials=self.n_trials,
            callbacks=[mlflow_callback]
        )
        
        # Get best parameters
        self.best_params = study.best_params
        self.best_score = study.best_value
        
        logger.info("Optimization complete!")
        logger.info(f"Best {self.optimization_metric}: {self.best_score:.4f}")
        logger.info(f"Best parameters: {self.best_params}")
        
        # Train final model with best parameters and log to MLflow
        best_run_id = self._train_best_model(X_train, X_test, y_train, y_test)
        
        return self.best_pipeline, best_run_id, study
    

    def _train_best_model(self, X_train, X_test, y_train, y_test):
        """Train the final model with best parameters and comprehensive MLflow logging."""

        with self.mlflow_setup.start_run(run_name=f"best_model_{self.model_class.__name__}") as run:
            # Log basic parameters
            self.mlflow_setup.log_param("test_size", self.test_size)
            self.mlflow_setup.log_param("model_type", self.model_class.__name__)
            self.mlflow_setup.log_param("n_numeric_features", len(self.numeric_features))
            self.mlflow_setup.log_param("n_categorical_features", len(self.categorical_features))
            self.mlflow_setup.log_param("target_column", self.target_column)
            self.mlflow_setup.log_param("n_trials", self.n_trials)
            self.mlflow_setup.log_param("optimization_metric", self.optimization_metric)

            # Log best hyperparameters
            for param_name, param_value in self.best_params.items():
                self.mlflow_setup.log_param(f"best_{param_name}", param_value)

            # Log fixed parameters if any
            for param_name, param_value in self.model_params.items():
                self.mlflow_setup.log_param(f"fixed_{param_name}", param_value)

            # Log feature names
            self.mlflow_setup.log_param("numeric_features", ", ".join(self.numeric_features))
            self.mlflow_setup.log_param("categorical_features", ", ".join(self.categorical_features))

            # Log dataset information
            self.mlflow_setup.log_param("n_train_samples", len(X_train))
            self.mlflow_setup.log_param("n_test_samples", len(X_test))

            # Create model with best parameters
            all_params = {**self.model_params, **self.best_params}
            self.best_model = self.model_class(**all_params)
            
            # Create and train pipeline
            self.best_pipeline = self.create_pipeline_train(self.best_model)
            self.best_pipeline.fit(X_train, y_train)
            
            # Make predictions
            y_train_pred = self.best_pipeline.predict(X_train)
            y_test_pred = self.best_pipeline.predict(X_test)
            
            # Calculate and log metrics
            train_metrics = self._calculate_metrics(y_train, y_train_pred, prefix="train")
            test_metrics = self._calculate_metrics(y_test, y_test_pred, prefix="test")
            
            all_metrics = {**train_metrics, **test_metrics}
            for metric_name, metric_value in all_metrics.items():
                self.mlflow_setup.log_metric(metric_name, metric_value)

            # Log the optimization score from Optuna
            self.mlflow_setup.log_metric(f"optuna_best_{self.optimization_metric}", self.best_score)

            # Log the model
            self.mlflow_setup.sklearn.log_model(
                self.best_pipeline,
                "model",
                input_example=X_train.iloc[:5],
                signature=self.mlflow_setup.models.infer_signature(X_train, y_train_pred),
                registered_model_name=self.mlflow_registered_model_name
            )

            if self.mlflow_registered_model_name:
                logger.info(f"Model successfully registered in Model Registry as: '{self.mlflow_registered_model_name}'")
            
            # Get the run ID for reference
            run_id = run.info.run_id
            
            logger.info(f"Best Model MLflow Run ID: {run_id}")
            logger.info(f"Tracking URI: {self.mlflow_setup.get_tracking_uri()}")
            logger.info(f"Train Accuracy: {train_metrics['train_accuracy']:.4f}")
            logger.info(f"Test Accuracy: {test_metrics['test_accuracy']:.4f}")
            
            return run_id
        
    def train(self):
        """Alias for train_with_optuna to maintain compatibility."""
        return self.train_with_optuna()
    

    def _calculate_metrics(self, y_true, y_pred, prefix=""):
        """
        Calculate classification metrics.

        Args:
            y_true: True labels
            y_pred: Predicted labels
            prefix: Prefix for metric names (e.g., "train" or "test")

        Returns:
            Dictionary of metrics
        """
        metrics = {}

        # Calculate metrics
        accuracy = accuracy_score(y_true, y_pred)
        metrics[f"{prefix}_accuracy"] = accuracy

        # For binary and multiclass classification
        try:
            precision = precision_score(
                y_true, y_pred, average='weighted', zero_division=0
            )
            recall = recall_score(
                y_true, y_pred, average='weighted', zero_division=0
            )
            f1 = f1_score(
                y_true, y_pred, average='weighted', zero_division=0
            )

            metrics[f"{prefix}_precision"] = precision
            metrics[f"{prefix}_recall"] = recall
            metrics[f"{prefix}_f1"] = f1
        except Exception:
            # In case of any issues with multiclass metrics
            pass

        return metrics

In [0]:
from sklearn.ensemble import RandomForestClassifier
test_size = 0.2

# Define RandomForest parameter distributions
param_distributions = {
    'n_estimators': ('int', 50, 200),
    'max_depth': ('int', 5, 30),
    'min_samples_split': ('int', 2, 10),
    'min_samples_leaf': ('int', 1, 5),
    'max_features': ('categorical', ['sqrt', 'log2', None])
}

trainer = TrainModel(
    df_fe,
    numeric_features = numeric_features,
    categorical_features = categorical_features,
    target_column = target_variable,
    model_class = RandomForestClassifier,
    test_size = test_size,
    model_params = {'random_state': 42, 'n_jobs': -1},
    param_distributions = param_distributions,
    n_trials = 10,
    optimization_metric = 'f1',
    mlflow_setup = mlflow,
    mlflow_registered_model_name = registered_model_rf,
    mlflow_experiment_path = experiment_path
)

best_pipeline, run_id, study = trainer.train()

In [0]:
from xgboost import XGBClassifier

# Define XGBoost parameter distributions
param_distributions_xgb = {
    'n_estimators': ('int', 100, 1000),
    'max_depth': ('int', 3, 10),
    'learning_rate': ('float', 0.01, 0.3, True), # Log scale es bueno para learning rate
    'subsample': ('float', 0.5, 1.0),
    'colsample_bytree': ('float', 0.5, 1.0),
    'gamma': ('float', 0, 0.5),
    'reg_alpha': ('float', 1e-5, 1.0, True), # L1 regularization
    'reg_lambda': ('float', 1e-5, 1.0, True)  # L2 regularization
}

fixed_model_params_xgb = {
    'random_state': 42, 
    'n_jobs': -1,
    'objective': 'binary:logistic', # clasificación binaria
    'eval_metric': 'logloss' 
}

trainer_xgb = TrainModel(
    df_fe,
    numeric_features = numeric_features,
    categorical_features = categorical_features,
    target_column = target_variable,
    model_class = XGBClassifier,  # <-- Cambio clave
    test_size = test_size,
    model_params = fixed_model_params_xgb, # <-- Cambio clave
    param_distributions = param_distributions_xgb, # <-- Cambio clave
    n_trials = 10, # Puedes ajustar esto
    optimization_metric = 'roc_auc', # 'roc_auc' es excelente para XGBoost
    mlflow_setup = mlflow,
    mlflow_registered_model_name = registered_model_xgb,  # Nombre para el registro del modelo
    mlflow_experiment_path = experiment_path
)

best_pipeline_xgb, run_id_xgb, study_xgb = trainer_xgb.train()

print("\n--- Entrenamiento de XGBoost completado ---")
print(f"Mejor Pipeline (XGB): {best_pipeline_xgb}")
print(f"MLflow Run ID (XGB): {run_id_xgb}")

In [0]:
from mlflow.exceptions import MlflowException
from mlflow.tracking import MlflowClient

mlflow_client = MlflowClient()
model_name = registered_model_rf
production_alias = "champion"

# Get the latest version of the registered model
all_versions = mlflow_client.search_model_versions(f"name='{model_name}'")
latest_version = max(int(v.version) for v in all_versions)
latest_model = mlflow_client.get_model_version(model_name, str(latest_version))

# Retrieve F1 score for the latest version
run = mlflow_client.get_run(latest_model.run_id)
latest_f1 = run.data.metrics.get("f1_val") or run.data.metrics.get("f1")

# Retrieve F1 score for the current champion version (if it exists)
try:
    champion_model = mlflow_client.get_model_version_by_alias(model_name, production_alias)
    champ_run = mlflow_client.get_run(champion_model.run_id)
    champ_f1 = champ_run.data.metrics.get("f1_val") or champ_run.data.metrics.get("f1")
except MlflowException:
    # Not existing champion alias; treat irs F1 as the lowest possible
    champ_f1 = float("-inf")

print(f"latest version: {latest_version}, F1 score: {latest_f1:.4f}")
print(f"champion version: {champion_model.version}, F1 score: {champ_f1:.4f}")

# Promote to champion only if it outperfoms the current champion
if latest_f1 > champ_f1:
    mlflow_client.set_registered_model_alias(
        name=model_name,
        version=str(latest_version),
        alias=production_alias,
    )
    mlflow_client.delete_registered_model_alias(model_name, "challenger")
    print(f"Promoted version {latest_version} to '{production_alias}' (F1 {latest_f1: .4f} > {champ_f1:.4f})")
else:
    print(f"Skipping promotion: latest F1 {latest_f1:.4f} <= champion F1 {champ_f1:.4f}")